# Missingness and Dirac Tradeoffs in LatentStateBuilder Inference

This notebook explains the practical missingness options for explicit latent-state workflows.

The new split is:

- `dsx.simulate(...)` / `Simulator` for generation, and
- `LatentStateBuilder` plus `dsx.path_log_prob(...)` for explicit latent-state inference.

The missingness machinery from the old simulator-conditioning path still matters here, especially when we compare regular Gaussian observations with `DiracIdentityObservation`.

## Two observation regimes

### 1. Standard noisy observations

For Gaussian / Student-t / independent observation models, the observation score is evaluated row by row using the repository's NaN-aware masking utilities.

### 2. `DiracIdentityObservation`

This is the exact-state regime `y_k = x_k`.

When a row is fully observed, there is no free latent variable to infer there: the state is fixed by the data. The payoff is a much smaller latent dimension. The cost is that the model becomes less forgiving: exact-state assumptions are only appropriate when that approximation is scientifically acceptable.

## Scan versus vmap intuition

There are two separate performance choices to keep in mind:

- `LatentStateBuilder` builds the latent path sequentially, because every transition depends on the previous state.
- `dsx.path_log_prob(..., chunk_size=...)` can score long trajectories as a **scan of vmaps** instead of one huge vmap.

So the transition structure is inherently sequential, but the observation and transition scoring can still be chunked to control memory use on long paths.

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
from numpyro.handlers import seed, trace

import dynestyx as dsx


## A discrete-time example with exact-state observations

In [ ]:
def dirac_model(obs_times=None, obs_values=None, predict_times=None):
    dynamics = dsx.DynamicalModel(
        control_dim=0,
        initial_condition=dist.MultivariateNormal(jnp.zeros(2), 0.4 * jnp.eye(2)),
        state_evolution=dsx.LinearGaussianStateEvolution(
            A=jnp.array([[0.72, 0.08], [0.0, 0.65]]),
            cov=jnp.array([[0.05, 0.01], [0.01, 0.04]]),
        ),
        observation_model=dsx.DiracIdentityObservation(),
    )
    return dsx.sample(
        "f",
        dynamics,
        obs_times=obs_times,
        obs_values=obs_values,
        predict_times=predict_times,
    )


times = jnp.arange(10.0)
with dsx.DiscreteTimeSimulator():
    with trace() as tr, seed(rng_seed=jr.PRNGKey(0)):
        dirac_model(predict_times=times)
obs_values = tr["f_observations"]["value"][0]
obs_values = obs_values.at[3].set(jnp.nan)
obs_values = obs_values.at[7].set(jnp.nan)


With `LatentStateBuilder`, only the fully missing rows need free latent variables. Fully observed rows are pinned directly by the exact-state observations.

In [ ]:
with dsx.LatentStateBuilder():
    with trace() as tr, seed(rng_seed=jr.PRNGKey(1)):
        dirac_model(obs_times=times, obs_values=obs_values)

sorted(name for name, site in tr.items() if site.get("type") == "sample" and "_x_" in name)


## Benefit / sacrifice summary

### Benefit of `DiracIdentityObservation`

- fewer free latent variables,
- simpler transition-dominated scoring,
- often much faster joint state + parameter inference.

### Sacrifice

- exact-state assumptions are stronger than noisy-observation assumptions,
- partially observed rows are less natural,
- and model mismatch is pushed into the latent dynamics rather than absorbed by observation noise.

When the approximation is scientifically acceptable, the dimension reduction can be substantial. When it is not, the standard noisy-observation path is the safer choice.

## Practical rule of thumb

- Use `dsx.path_log_prob(..., chunk_size=...)` when you want standalone path scoring with explicit memory control.
- Use `LatentStateBuilder` when you want explicit latent sites inside NumPyro.
- Reach for `DiracIdentityObservation` only when the exact-state approximation is genuinely appropriate and the reduced latent dimension is worth the modeling tradeoff.